In [7]:

import os
from openai import OpenAI
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
import base64

from email.utils import parseaddr

In [2]:
from pathlib import Path


SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]


TOKEN_PATH = Path(r"x:\project\email-auto-responser\token.json")

creds = Credentials.from_authorized_user_file(
    str(TOKEN_PATH),
    SCOPES
)

In [11]:
gmail_service = build("gmail", "v1", credentials=creds)

def fetch_gmail(gmail_service):
    
    results = gmail_service.users().messages().list(
        userId="me",
        q="category:primary is:unread",
        maxResults=1
    ).execute()


    data = results.get("messages", [])
    # return fetch_gmail(gmail_service)
    print(data)

    if data:
        message_id = data[0]["id"]
    else:
        return None

    message = gmail_service.users().messages().get(
        userId="me",
        id=message_id,
        format ="full"
    ).execute()

    payload = message.get("payload", {})

    headers = payload.get("headers", [])

    sender_email = parseaddr(
            next(
            header["value"]
            for header in headers
            if header["name"] == "From"
        )
    )[1]

    parts = payload.get("parts", [])

    text = ""
    if parts:
        for part in parts:
            if part.get("mimeType") == "text/plain":
                message_content = part.get("body", {}).get("data")
            
                if message_content:
                    decode_message = base64.urlsafe_b64decode(message_content)
                    text = decode_message.decode("utf-8")


                    break

                    # gmail_service.users().messages().modify(
                    #     userId="me",
                    #     id=message_id,
                    #     body= {
                    #         "removeLabelIds": ["UNREAD"]
                    #     }
                    # ).execute()
    # else:
    #     message_content = payload.get("body", {}).get("data")
    #     if message_content:
    #         decode_message = base64.urlsafe_b64decode(message_content)
    #         text = decode_message.decode("utf-8")

                   


    return text, sender_email





email = fetch_gmail(gmail_service)

if email:
    print(email)
else:
    print("No unread emails found")

[{'id': '19ddd5128a5cfb43', 'threadId': '19ddd5128a5cfb43'}]
('', 'no-reply@e.udemymail.com')


In [5]:
email = fetch_gmail(gmail_service)

if email:
    print(email)
else:
    print("No unread emails found")

[{'id': '19ddd5128a5cfb43', 'threadId': '19ddd5128a5cfb43'}]
('', 'Udemy <no-reply@e.udemymail.com>')


In [97]:
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

embeddings = OpenAIEmbeddings(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="perplexity/pplx-embed-v1-0.6b",
    check_embedding_ctx_length=False,
)

In [99]:
vector_db = QdrantVectorStore.from_existing_collection(
    url="http://localhost:6333",
    collection_name="company-database",
    embedding=embeddings,
    validate_collection_config=False,
)

In [ ]:
def get_query(email: str):

    search_result = vector_db.similarity_search(
        query=email)


    context = "\n\n\n".join([f"""
        page_content:{result.page_content}"""
        for result in search_result
        ])
    
    system_prompt = f"""You are a professional email responser agent that give reply to mails
    only from provided context
    
    if mail doesn't belong from or relate from context then say i will share this with support team, Be paitent

    and if the mail have too many link reply that as spam


    context: {context}
    """

    responser = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[

            {"role":"system", "content": system_prompt},
            {"role":"user", "content": email},
 
            ]
        
    )

    return responser.choices[0].message.content






In [100]:
email = fetch_gmail(gmail_service)

if email:
    reply = get_query(email)


[{'id': '19e7169b6f79fd56', 'threadId': '19e61d077d335a51'}]
[Document(metadata={'source': 'X:\\project\\email-auto-responser\\company-document-text.csv', 'row': 1005, '_id': '7147308f-f0ef-43b1-81af-9956d7547e1d', '_collection_name': 'company-database'}, page_content='text: order id  10716 shipping details  ship name  rancho grande ship address  av. del libertador 900 ship city  buenos aires ship region  south america ship postal code  1010 ship country  argentina customer details  customer id  ranch customer name  rancho grande employee details  employee name  margaret peacock shipper details  shipper id  2 shipper name  united package order details  order date  2017-10-24 shipped date  2017-10-27 products  -------------------------------------------------------------------------------------------------- product  sir rodney s scones quantity  5 unit price  10 0 total  50 0 -------------------------------------------------------------------------------------------------- product  manj

In [101]:

print(reply)

Subject: Re: Question Regarding Shipment for Order #10448

Dear Rancho Grande,

Thank you for reaching out. Here is what our records show for Order #10448:

- Order Date: 2017-02-17
- Shipped Date: 2017-02-24
- Shipper: United Package (Shipper ID 2)
- Shipping To: Rancho Grande, Av. del Libertador 900, Buenos Aires, 1010, Argentina
- Employee Handling: Margaret Peacock
- Items:
  - gumbär gummibärchen — 6 @ 24.90 each — total 149.40
  - boston crab meat — 20 @ 14.70 each — total 294.00

Status:
- The order was shipped on 2017-02-24. Our system does not show a final delivery confirmation in this view. I will escalate this to our carrier to obtain the latest delivery status and any available tracking information.

What I will do next:
- I will contact United Package to obtain the current delivery status and a tracking number (if available).
- I will share an updated estimated delivery window as soon as I have the carrier’s information.
- If the shipment has already been delivered, I will

In [ ]:
response = client.embeddings.create(
    model="perplexity/pplx-embed-v1-0.6b",
    input=["test"]
)
print(response)

CreateEmbeddingResponse(data=[Embedding(embedding=[0.0, -0.1640625, 0.0, -0.515625, 0.1640625, 0.4765625, -0.34375, 0.171875, -0.28125, -0.1328125, -0.2734375, -0.7734375, 0.4453125, 0.0, -0.171875, -0.078125, -0.8515625, -0.1171875, 0.046875, -0.2734375, 0.390625, -0.46875, -0.1484375, 0.28125, 0.015625, 0.375, 0.1953125, -0.71875, -0.046875, 0.0, 0.484375, -0.5, -0.765625, 0.21875, 0.0390625, 0.0, -0.0703125, 0.4140625, 0.109375, -0.3046875, -0.1640625, -0.2578125, 0.171875, -0.0546875, 0.3828125, 0.390625, -0.203125, -0.6015625, -0.0234375, 0.15625, 0.4453125, 0.0625, 0.140625, 0.171875, -0.171875, -0.09375, 0.6328125, -0.3046875, -0.234375, 0.1328125, -0.2890625, -0.0546875, -0.703125, 0.1875, -0.0390625, -0.0390625, -0.1875, -0.390625, -0.296875, -0.4453125, -0.4921875, -0.5859375, 0.1875, 0.375, -0.515625, 0.296875, 0.171875, -0.0625, -0.1328125, -0.234375, -0.0703125, 0.3828125, 0.2265625, 0.078125, 0.203125, 0.4140625, -0.453125, 0.0859375, -0.0234375, -0.390625, 0.1796875, 0.4

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

response = client.embeddings.create(
    model="perplexity/pplx-embed-v1-0.6b",
    input="hello world"
)

embedding = response.data[0].embedding

print(len(embedding))

1024
